# Process Plant Electrical Load List

Demonstrates plant-wide electrical load analysis using NeqSim.
After running a process simulation, the `ProcessSystem` automatically
aggregates every driven equipment's electrical demand into a
structured load list per IEC 61936.

**Process:** Gas receiving, separation, compression, pumping

**Outputs:**
- Total connected load (kW / kVA)
- Maximum demand with diversity factors
- Required transformer and generator sizing
- Overall plant power factor
- Load breakdown by equipment type

In [ ]:
# Setup
import subprocess, sys, pathlib, os

try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False)
    if ns is not None:
        ns = neqsim_classes(ns)
        NEQSIM_MODE = "devtools"
        print("NeqSim loaded via devtools")
    else:
        raise RuntimeError("devtools returned None")
except Exception:
    try:
        import neqsim
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
    print("NeqSim loaded via pip")

import json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
Compressor = jneqsim.process.equipment.compressor.Compressor
Pump = jneqsim.process.equipment.pump.Pump
Cooler = jneqsim.process.equipment.heatexchanger.Cooler
Separator = jneqsim.process.equipment.separator.Separator
ThreePhaseSeparator = jneqsim.process.equipment.separator.ThreePhaseSeparator
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

print(f"Mode: {NEQSIM_MODE}")

## 1. Build a Multi-Equipment Process

The process includes:
- **HP Separator** — 3-phase inlet separation at 60 bara
- **Gas Compressor** — boost gas from 60 to 120 bara
- **Gas Cooler** — cool compressed gas
- **Export Pump** — pump condensate to 80 bara
- **Booster Compressor** — further boost to 180 bara

In [ ]:
# Create well fluid (gas-condensate)
fluid = SystemSrkEos(273.15 + 80.0, 60.0)
fluid.addComponent("methane", 0.70)
fluid.addComponent("ethane", 0.10)
fluid.addComponent("propane", 0.05)
fluid.addComponent("i-butane", 0.02)
fluid.addComponent("n-butane", 0.02)
fluid.addComponent("n-pentane", 0.01)
fluid.addComponent("n-hexane", 0.01)
fluid.addComponent("nitrogen", 0.03)
fluid.addComponent("CO2", 0.04)
fluid.addComponent("water", 0.02)
fluid.setMixingRule("classic")
fluid.setMultiPhaseCheck(True)

# Build process
process = ProcessSystem()

# Feed stream
feed = Stream("Well Stream", fluid)
feed.setFlowRate(80000.0, "kg/hr")
process.add(feed)

# HP Separator
hp_sep = Separator("HP Separator", feed)
process.add(hp_sep)

# Gas compression — Stage 1
gas_comp1 = Compressor("Gas Compressor", hp_sep.getGasOutStream())
gas_comp1.setOutletPressure(120.0)
process.add(gas_comp1)

# Gas aftercooler
gas_cooler = Cooler("Gas Aftercooler", gas_comp1.getOutletStream())
gas_cooler.setOutTemperature(273.15 + 40.0)
process.add(gas_cooler)

# Booster compressor — Stage 2
gas_comp2 = Compressor("Booster Compressor", gas_cooler.getOutletStream())
gas_comp2.setOutletPressure(180.0)
process.add(gas_comp2)

# Export pump on condensate
export_pump = Pump("Export Pump", hp_sep.getLiquidOutStream())
export_pump.setOutletPressure(80.0)
process.add(export_pump)

# Run process
process.run()

print("=== Process Equipment Duty ===")
print(f"Gas Compressor:     {gas_comp1.getPower('kW'):.1f} kW")
print(f"Booster Compressor: {gas_comp2.getPower('kW'):.1f} kW")
print(f"Export Pump:        {export_pump.getPower('kW'):.1f} kW")

## 2. Run Electrical Designs & Generate Load List

The `ProcessSystem.getElectricalLoadList()` method:
1. Runs `calcDesign()` on each equipment's electrical design
2. Creates a `LoadItem` for each driven equipment
3. Aggregates into an `ElectricalLoadList` with demand/diversity factors

$$P_{\text{max,demand}} = \sum_i P_{\text{absorbed},i} \times f_{\text{demand},i} \times f_{\text{diversity},i}$$

In [ ]:
# Run all electrical designs
process.runAllElectricalDesigns()

# Get the plant load list
load_list = process.getElectricalLoadList()

print("=== Electrical Load List Summary ===")
print(f"Number of loads:          {load_list.getLoadCount()}")
print(f"Total connected load:     {load_list.getTotalConnectedLoadKW():.1f} kW")
print(f"Total connected load:     {load_list.getTotalConnectedLoadKVA():.1f} kVA")
print(f"Maximum demand:           {load_list.getMaximumDemandKW():.1f} kW")
print(f"Maximum demand:           {load_list.getMaximumDemandKVA():.1f} kVA")
print(f"Overall power factor:     {load_list.getOverallPowerFactor():.3f}")
print(f"Required transformer:     {load_list.getRequiredTransformerKVA():.0f} kVA")
print(f"Required generator:       {load_list.getRequiredGeneratorKVA():.0f} kVA")

In [ ]:
# Print full JSON load list
load_json = json.loads(str(load_list.toJson()))
print(json.dumps(load_json, indent=2))

## 3. Load Breakdown Visualization

In [ ]:
# Extract load items from JSON
items = load_json.get("loadItems", [])
if not items:
    print("No electrical loads found — equipment may not have power consumers.")
else:
    names = [item["tagNumber"] for item in items]
    rated_kw = [item["ratedPowerKW"] for item in items]
    demand_kw = [item["maxDemandKW"] for item in items]
    voltages = [item.get("ratedVoltageV", 0) for item in items]

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Bar chart — rated vs demand
    ax = axes[0]
    x = np.arange(len(names))
    w = 0.35
    ax.bar(x - w/2, rated_kw, w, label="Rated (kW)", color="steelblue")
    ax.bar(x + w/2, demand_kw, w, label="Max Demand (kW)", color="coral")
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha="right", fontsize=9)
    ax.set_ylabel("Power (kW)")
    ax.set_title("Rated Power vs Max Demand")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis="y")

    # Pie chart — load share
    ax = axes[1]
    ax.pie(demand_kw, labels=names, autopct="%1.1f%%", startangle=90,
           colors=plt.cm.Set3.colors)
    ax.set_title("Load Share (by Max Demand)")

    # Summary bar
    ax = axes[2]
    summary = load_json.get("summary", {})
    labels = ["Connected\n(kW)", "Connected\n(kVA)", "Max Demand\n(kW)",
              "Max Demand\n(kVA)", "Req. Xfmr\n(kVA)"]
    values = [
        summary.get("totalConnectedLoadKW", 0),
        summary.get("totalConnectedLoadKVA", 0),
        summary.get("maximumDemandKW", 0),
        summary.get("maximumDemandKVA", 0),
        summary.get("requiredTransformerKVA", 0),
    ]
    colors = ["#4e79a7", "#59a14f", "#f28e2b", "#e15759", "#76b7b2"]
    ax.bar(labels, values, color=colors)
    for i, v in enumerate(values):
        ax.text(i, v + max(values)*0.02, f"{v:.0f}", ha="center", fontsize=10, fontweight="bold")
    ax.set_ylabel("Power")
    ax.set_title("Plant Electrical Summary")
    ax.grid(True, alpha=0.3, axis="y")

    plt.tight_layout()
    plt.show()

## 4. Equipment-Level Comparison Table

In [ ]:
if items:
    header = f"{'Tag':<22} {'Type':<14} {'Rated':>8} {'Demand':>8} {'PF':>6} {'Voltage':>8} {'VFD':>5}"
    print(header)
    print("-" * len(header))
    for item in items:
        print(f"{item['tagNumber']:<22} "
              f"{item['description']:<14} "
              f"{item['ratedPowerKW']:>8.1f} "
              f"{item['maxDemandKW']:>8.1f} "
              f"{item.get('powerFactor', 0):>6.3f} "
              f"{item.get('ratedVoltageV', 0):>8.0f} "
              f"{'Yes' if item.get('hasVFD') else 'No':>5}")

    summary = load_json.get("summary", {})
    print(f"\n{'TOTAL':<22} {'':14} "
          f"{summary.get('totalConnectedLoadKW', 0):>8.1f} "
          f"{summary.get('maximumDemandKW', 0):>8.1f} "
          f"{summary.get('overallPowerFactor', 0):>6.3f}")
else:
    print("No loads to display.")

## 5. Individual Equipment Electrical Report

In [ ]:
# Get the gas compressor's detailed electrical design
ed_comp = process.getEquipmentElectricalDesign("Gas Compressor")
if ed_comp is not None:
    report = json.loads(str(ed_comp.toJson()))
    print("=== Gas Compressor — Full Electrical Report ===")
    print(json.dumps(report, indent=2))
else:
    print("No electrical design for Gas Compressor")

## 6. Transformer Sizing from Load List

The load list provides recommended transformer and generator sizes:

$$S_{\text{transformer}} = S_{\text{max,demand}} \times 1.15$$

$$S_{\text{generator}} = S_{\text{max,demand}} \times 1.15 \times 1.10$$

In [ ]:
# Size a transformer for the plant
import jpype
Transformer = jpype.JClass("neqsim.process.electricaldesign.components.Transformer")

xfmr = Transformer()
total_kva = load_list.getMaximumDemandKVA()
xfmr.sizeTransformer(float(total_kva), 11000.0, 400.0)

print("=== Main Power Transformer ===")
print(f"Plant max demand:    {total_kva:.0f} kVA")
print(f"Transformer rating:  {xfmr.getRatedPowerKVA():.0f} kVA")
print(f"Primary voltage:     {xfmr.getPrimaryVoltageV():.0f} V")
print(f"Secondary voltage:   {xfmr.getSecondaryVoltageV():.0f} V")
print(f"Efficiency:          {xfmr.getEfficiencyPercent():.1f}%")
print(f"Impedance:           {xfmr.getImpedancePercent():.1f}%")
print(f"Total losses:        {xfmr.getTotalLossKW():.1f} kW")
print(f"Cooling:             {xfmr.getCoolingType()}")
print(f"Vector group:        {xfmr.getVectorGroup()}")

xfmr_report = json.loads(str(xfmr.toJson()))
print("\nFull transformer report:")
print(json.dumps(xfmr_report, indent=2))

## Summary

This notebook demonstrated:

1. **Plant-wide load aggregation** from process simulation results
2. **Load list** with demand/diversity factors per IEC 61936
3. **Load breakdown** — pie charts, bar charts, tabular format
4. **Transformer sizing** from aggregated load list
5. **JSON export** of load list and individual equipment

The load list formula:

$$P_{\text{demand}} = \sum_i P_i \times f_{\text{demand},i} \times f_{\text{diversity},i}$$

See also:
- `compressor_electrical_design.ipynb` — compressor motor + cable sizing
- `motor_vfd_analysis.ipynb` — VFD topology, harmonics, part-load curves